%md
# 02 - Simulate New Orders
Drops a small batch CSV into the orders landing folder to simulate streaming arrivals.
Each batch includes 3 new valid orders, 1 duplicate of an existing order, and 1 bad row (null order_id) to test Silver-layer data quality.

In [0]:
import uuid
import pandas as pd
from datetime import datetime, timedelta

catalog = "retail_lakehouse"
orders_folder = f"/Volumes/{catalog}/bronze/raw_files/orders"

columns = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]

# Reuse real customers so joins work later
customer_ids = [r.customer_id for r in
                spark.table(f"{catalog}.bronze.customers_raw").select("customer_id").limit(3).collect()]

now = datetime.now()
fmt = "%Y-%m-%d %H:%M:%S"
rows = []

# 3 new valid orders
for cid in customer_ids:
    rows.append({
        "order_id": uuid.uuid4().hex,
        "customer_id": cid,
        "order_status": "approved",
        "order_purchase_timestamp": now.strftime(fmt),
        "order_approved_at": (now + timedelta(minutes=10)).strftime(fmt),
        "order_delivered_carrier_date": None,
        "order_delivered_customer_date": None,
        "order_estimated_delivery_date": (now + timedelta(days=10)).strftime(fmt),
    })

# 1 duplicate of an existing order (Silver should remove it)
existing = spark.table(f"{catalog}.bronze.orders_raw").select(*columns).limit(1).toPandas()
rows.append(existing.iloc[0].to_dict())

# 1 bad row with no order_id (a Silver expectation should drop it)
rows.append({**rows[0], "order_id": None})

batch_df = pd.DataFrame(rows, columns=columns)
file_path = f"{orders_folder}/orders_batch_{now.strftime('%Y%m%d_%H%M%S')}.csv"
batch_df.to_csv(file_path, index=False)

print(f"Wrote {len(batch_df)} rows to {file_path}")
display(batch_df)

## Simulate late-arriving updates
Writes 2 files for one order:
1. **Update** → order is now `delivered` (newer event)
2. **Stale record** → an old `created` version that arrives *after* the update

The Silver pipeline must keep `delivered`, because it orders by event time, not arrival time.%md


In [0]:
import time
import pandas as pd
from datetime import timedelta
from pyspark.sql import functions as F

catalog = "retail_lakehouse"
orders_folder = f"/Volumes/{catalog}/bronze/raw_files/orders"
fmt = "%Y-%m-%d %H:%M:%S"
columns = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]

# Pick one of the simulated new orders that's still 'approved'
target = (
    spark.table(f"{catalog}.silver.orders")
    .filter(F.col("_source_file").contains("orders_batch_"))
    .filter("order_status = 'approved'")
    .limit(1)
    .toPandas()
    .iloc[0]
)
purchase = pd.Timestamp(target.order_purchase_timestamp)
print("Updating order:", target.order_id)

# File 1: newer update -> delivered
update_row = {
    "order_id": target.order_id,
    "customer_id": target.customer_id,
    "order_status": "delivered",
    "order_purchase_timestamp": purchase.strftime(fmt),
    "order_approved_at": (purchase + timedelta(minutes=10)).strftime(fmt),
    "order_delivered_carrier_date": (purchase + timedelta(days=1)).strftime(fmt),
    "order_delivered_customer_date": (purchase + timedelta(days=3)).strftime(fmt),
    "order_estimated_delivery_date": (purchase + timedelta(days=10)).strftime(fmt),
}
pd.DataFrame([update_row], columns=columns).to_csv(
    f"{orders_folder}/orders_update_{int(time.time())}.csv", index=False)

time.sleep(2)

# File 2: stale old version arriving LATE -> must NOT win
stale_row = {
    **update_row,
    "order_status": "created",
    "order_approved_at": None,
    "order_delivered_carrier_date": None,
    "order_delivered_customer_date": None,
}
pd.DataFrame([stale_row], columns=columns).to_csv(
    f"{orders_folder}/orders_late_stale_{int(time.time())}.csv", index=False)

print("✅ Wrote update file, then stale late file")